In [1]:
import os
import re
import pandas as pd
from difflib import get_close_matches
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, LpBinary, LpStatus

In [ ]:
import os
from tkinter import Tk, filedialog

BASE_FOLDER = r"C:\Users\chompoopan.jan\OneDrive - Charoen Pokphand Foods Group\Documents\workforce_system"

INPUT_FOLDER = os.path.join(BASE_FOLDER, "Input")
OUTPUT_FOLDER = os.path.join(BASE_FOLDER, "Output")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

master_files = [
    f for f in os.listdir(INPUT_FOLDER)
    if f.lower().endswith((".xlsx", ".xls")) and ("active" in f.lower() or "master" in f.lower())
]

manpower_files = [
    f for f in os.listdir(INPUT_FOLDER)
    if f.lower().endswith((".xlsx", ".xls")) and "manpower" in f.lower()
]

skill_files = [
    f for f in os.listdir(INPUT_FOLDER)
    if f.lower().endswith((".xlsx", ".xls")) and ("skill" in f.lower() or "matrix" in f.lower())
]

MASTER_FILE = os.path.join(INPUT_FOLDER, master_files[0])
MANPOWER_FILE = os.path.join(INPUT_FOLDER, manpower_files[0])
SKILL_FILE = os.path.join(INPUT_FOLDER, skill_files[0])

# popup เลือกไฟล์ timestamp
root = Tk()
root.withdraw()
root.attributes("-topmost", True)

SCAN_FILE = filedialog.askopenfilename(
    title="เลือกไฟล์ Timestamp / Time Record",
    filetypes=[
        ("Excel or CSV files", "*.csv *.xlsx *.xls"),
        ("CSV files", "*.csv"),
        ("Excel files", "*.xlsx *.xls"),
        ("All files", "*.*")
    ]
)

root.destroy()

if not SCAN_FILE:
    raise FileNotFoundError("ยังไม่ได้เลือกไฟล์ timestamp")

print("SCAN_FILE:", SCAN_FILE)
print("MASTER_FILE:", MASTER_FILE)
print("MANPOWER_FILE:", MANPOWER_FILE)
print("SKILL_FILE:", SKILL_FILE)

SCAN_FILE: C:/Users/chompoopan.jan/OneDrive - Charoen Pokphand Foods Group/Documents/Time_Record_Export_19052026_081416.csv
MASTER_FILE: C:\Users\chompoopan.jan\OneDrive - Charoen Pokphand Foods Group\PARINYA THEPHI's files - PLP - Process Improvement (Production)\3_Step 3\Production System\workforce_system\Input\2569-04-24 - ( Active) -โรงงานชำแหละและตัดแต่งสุกร พระพุ.xlsx
MANPOWER_FILE: C:\Users\chompoopan.jan\OneDrive - Charoen Pokphand Foods Group\PARINYA THEPHI's files - PLP - Process Improvement (Production)\3_Step 3\Production System\workforce_system\Input\manpower.xlsx
SKILL_FILE: C:\Users\chompoopan.jan\OneDrive - Charoen Pokphand Foods Group\PARINYA THEPHI's files - PLP - Process Improvement (Production)\3_Step 3\Production System\workforce_system\Input\skill_metrix (1).xlsx


In [3]:
if SCAN_FILE.lower().endswith(".csv"):
    try:
        scan = pd.read_csv(SCAN_FILE, skiprows=10, encoding="utf-8-sig")
    except UnicodeDecodeError:
        scan = pd.read_csv(SCAN_FILE, skiprows=10, encoding="cp874")
else:
    scan = pd.read_excel(SCAN_FILE, skiprows=10)

scan.columns = scan.iloc[0]
scan = scan[1:].reset_index(drop=True)

scan = scan[["Timestamp", "Employee ID", "Employee Name"]].copy()

scan["Timestamp"] = pd.to_datetime(
    scan["Timestamp"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce"
)

scan = scan.dropna(subset=["Timestamp"])

scan["emp_id"] = scan["Employee ID"].astype(str).str.strip()
scan["date"] = scan["Timestamp"].dt.date
scan["time"] = scan["Timestamp"].dt.strftime("%H:%M")

TARGET_DATE = str(scan["date"].max())
LATE_GRACE_MINUTES = 5

print("Auto TARGET_DATE:", TARGET_DATE)

Auto TARGET_DATE: 2026-05-19


In [4]:
# =========================
# Cell 5: Read Master + Clean emp_id + Fallback + Rebuild Attendance
# =========================

import re
import pandas as pd
from difflib import get_close_matches

# -------------------------
# 1) Utility functions
# -------------------------

def clean_emp_id(x):
    if pd.isna(x):
        return ""

    x = str(x).replace("\u00a0", "").strip()
    x = re.sub(r"\s+", "", x)
    x = re.sub(r"\.0$", "", x)
    x = re.sub(r"[^0-9]", "", x)

    return x


def clean_name(x):
    if pd.isna(x):
        return ""

    x = str(x).replace("\u00a0", " ").strip()
    x = re.sub(r"\s+", "", x)
    x = re.sub(
        r"^(นาย|นางสาว|นาง|mr|mrs|miss|ms)",
        "",
        x,
        flags=re.IGNORECASE
    )

    return x


# ให้ cell อื่นใช้ชื่อ clean_id ได้ด้วย
clean_id = clean_emp_id


# -------------------------
# 2) Read master employee
# -------------------------

employee_raw = pd.read_excel(MASTER_FILE)

employee = employee_raw.rename(columns={
    "User ID (Job Information)": "emp_id",
    "First Name (Local)": "first_name",
    "Last Name (Local)": "last_name",
    "Employment Status": "status",
    "Name (Employment Type)": "employment_type",
    "Code (Position)": "position_code",
    "Title (Position)": "position",
    "หน่วยงาน": "dept",
    "Name (Section)": "section"
})

employee.columns = (
    employee.columns
    .astype(str)
    .str.strip()
)

required_master_cols = [
    "emp_id",
    "first_name",
    "last_name",
    "status",
    "employment_type",
    "position_code",
    "position",
    "dept",
    "section"
]

missing_cols = [
    col for col in required_master_cols
    if col not in employee.columns
]

if missing_cols:
    raise ValueError(
        f"Master employee ขาด column: {missing_cols}\n"
        f"Column ที่มีคือ: {employee.columns.tolist()}"
    )

employee["emp_id"] = employee["emp_id"].apply(clean_emp_id)

employee["name"] = (
    employee["first_name"].fillna("").astype(str).str.strip()
    + " "
    + employee["last_name"].fillna("").astype(str).str.strip()
)

employee["name_key"] = employee["name"].apply(clean_name)

employee = employee[
    [
        "emp_id",
        "name",
        "name_key",
        "status",
        "employment_type",
        "position_code",
        "position",
        "dept",
        "section"
    ]
].copy()


# -------------------------
# 3) Clean scan emp_id/name
# -------------------------

scan["emp_id"] = scan["Employee ID"].apply(clean_emp_id)
scan["name_key"] = scan["Employee Name"].apply(clean_name)


# -------------------------
# 4) Check mapping by emp_id
# -------------------------

test_merge = scan.merge(
    employee[["emp_id", "name", "dept", "section", "position", "name_key"]],
    on="emp_id",
    how="left",
    suffixes=("", "_master")
)

missing_before = test_merge[
    test_merge["dept"].isna()
].copy()

print("ก่อน fallback:")
print("จำนวน record map ไม่เจอ:", len(missing_before))
print("จำนวน emp_id map ไม่เจอ:", missing_before["emp_id"].nunique())


# -------------------------
# 5) Fallback ด้วยชื่อแบบ exact match
# -------------------------

name_lookup = (
    employee[
        ["emp_id", "name", "dept", "section", "position", "name_key"]
    ]
    .dropna(subset=["name_key"])
    .drop_duplicates("name_key")
)

name_to_emp = dict(
    zip(name_lookup["name_key"], name_lookup["emp_id"])
)

condition_missing = test_merge["dept"].isna()

scan.loc[
    condition_missing,
    "emp_id_by_name"
] = scan.loc[
    condition_missing,
    "name_key"
].map(name_to_emp)

condition_exact_name = (
    condition_missing &
    scan["emp_id_by_name"].notna()
)

scan.loc[
    condition_exact_name,
    "emp_id"
] = scan.loc[
    condition_exact_name,
    "emp_id_by_name"
]

print("จำนวน record ที่ fallback ด้วยชื่อได้:", condition_exact_name.sum())


# -------------------------
# 6) Check mapping after fallback
# -------------------------

test_merge_after = scan.merge(
    employee[["emp_id", "name", "dept", "section", "position", "name_key"]],
    on="emp_id",
    how="left",
    suffixes=("", "_master")
)

missing_after = test_merge_after[
    test_merge_after["dept"].isna()
].copy()

print("หลัง fallback:")
print("จำนวน record map ไม่เจอ:", len(missing_after))
print("จำนวน emp_id map ไม่เจอ:", missing_after["emp_id"].nunique())

print(
    missing_after[
        ["emp_id", "Employee Name", "name_key"]
    ]
    .drop_duplicates()
    .head(50)
)


# -------------------------
# 7) Rebuild attendance หลัง clean/fallback emp_id
# -------------------------

attendance = (
    scan.groupby(["emp_id", "Employee Name", "date"])
    .agg(
        scan_in=("time", "min"),
        scan_out=("time", "max")
    )
    .reset_index()
)

attendance = attendance.rename(columns={
    "Employee Name": "scan_name"
})

print("Rebuild attendance done")
print(attendance.head())

ก่อน fallback:
จำนวน record map ไม่เจอ: 8
จำนวน emp_id map ไม่เจอ: 7
จำนวน record ที่ fallback ด้วยชื่อได้: 0
หลัง fallback:
จำนวน record map ไม่เจอ: 8
จำนวน emp_id map ไม่เจอ: 7
       emp_id       Employee Name           name_key
123  10695551  PHITSANU KHAOTHONG  PHITSANUKHAOTHONG
157  10695547  SIRILUCK NETSAWANG  SIRILUCKNETSAWANG
181  10635325    PANADDA SRIWAREE    PANADDASRIWAREE
202                           NaN                   
256  10695574     THIDARAT PHOMMA     THIDARATPHOMMA
257  10687719     THIDARAT PHOMMA     THIDARATPHOMMA
262  10695445      JARATPORN SUPA      JARATPORNSUPA
Rebuild attendance done
     emp_id             scan_name        date scan_in scan_out
0  10249292        NAPAPA SALEAVE  2026-05-19   04:38    04:38
1  10290692  THIRATHIP NIYOMTHONG  2026-05-19   00:35    06:51
2  10291674       WARAPHON KHAOSI  2026-05-19   04:50    04:50
3  10296449        WANNA NILKOOHA  2026-05-19   02:03    02:03
4  10296630       OUYTIP JANGSONG  2026-05-19   04:56    0

In [5]:
# =========================
# Cell 5.2: เพิ่มหน่วยงานลงในไฟล์ timestamp
# =========================

employee_dept = employee[
    ["emp_id", "name", "dept", "section", "position"]
].copy()

scan_with_dept = scan.merge(
    employee_dept,
    on="emp_id",
    how="left"
)

scan_with_dept = scan_with_dept[
    [
        "Timestamp",
        "emp_id",
        "Employee Name",
        "name",
        "dept",
        "section",
        "position",
        "date",
        "time"
    ]
]

print(scan_with_dept.head())

            Timestamp    emp_id       Employee Name                name  \
0 2026-05-19 07:35:46  10604117             YUM EAM             YUM EAM   
1 2026-05-19 07:35:46  10604117             YUM EAM             YUM EAM   
2 2026-05-19 07:35:42  10604117             YUM EAM             YUM EAM   
3 2026-05-19 07:35:42  10604117             YUM EAM             YUM EAM   
4 2026-05-19 07:35:37  10656662  AMONNED NUPRASROED  อมรเณศ หนูประเสริฐ   

                       dept                               section  \
0  งานแยกชิ้นส่วนและตัดแต่ง  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
1  งานแยกชิ้นส่วนและตัดแต่ง  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
2  งานแยกชิ้นส่วนและตัดแต่ง  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
3  งานแยกชิ้นส่วนและตัดแต่ง  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
4  งานแยกชิ้นส่วนและตัดแต่ง  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   

                              position        date   time  
0     ผู้ปฏิบัติการ (ตัดแต่งหมูไหล่-D)  2026-05-19  07:35  
1     ผู้ปฏิบั

In [6]:
#6
manpower_raw = pd.read_excel(MANPOWER_FILE)

manpower = manpower_raw.rename(columns={
    "หน่วยงาน": "dept",
    "หน้างาน": "work_station",
    "กะ": "shift",
    "จำนวนคน": "required_hc",
    "เวลาเข้า": "shift_start",
    "เวลาออก": "shift_end"
})

# ลบแถวที่ไม่มีจำนวนคน หรือไม่มีหน้างาน
manpower = manpower.dropna(subset=["required_hc", "work_station"]).copy()

manpower["date"] = TARGET_DATE
manpower["required_skill"] = manpower["work_station"]

manpower["dept"] = manpower["dept"].astype(str).str.strip()
manpower["work_station"] = manpower["work_station"].astype(str).str.strip()
manpower["shift"] = manpower["shift"].astype(str).str.strip()

# แปลงจำนวนคนให้เป็นตัวเลขก่อน แล้วค่อยเป็น int
manpower["required_hc"] = pd.to_numeric(
    manpower["required_hc"],
    errors="coerce"
)

manpower = manpower.dropna(subset=["required_hc"]).copy()
manpower["required_hc"] = manpower["required_hc"].astype(int)

manpower["shift_start"] = pd.to_datetime(
    manpower["shift_start"].astype(str),
    errors="coerce"
).dt.strftime("%H:%M")

manpower["shift_end"] = pd.to_datetime(
    manpower["shift_end"].astype(str),
    errors="coerce"
).dt.strftime("%H:%M")

manpower_plan = manpower[
    [
        "date",
        "dept",
        "work_station",
        "shift",
        "required_hc",
        "shift_start",
        "shift_end",
        "required_skill"
    ]
].copy()

print(manpower_plan.head())

         date             dept   work_station shift  required_hc shift_start  \
0  2026-05-19         งานพัสดุ          พัสดุ  กะ 1            3       08:00   
1  2026-05-19      งานวิศวกรรม  วิศวะ กลางวัน  กะ 1            3       08:00   
2  2026-05-19      งานวิศวกรรม  วิศวะ กลางคืน  กะ 2            3       20:00   
3  2026-05-19  งานควบคุมคุณภาพ         Office  กะ 1            1       08:00   
4  2026-05-19  งานควบคุมคุณภาพ     QC ตัดแต่ง  กะ 1            7       08:00   

  shift_end  required_skill  
0     17:00           พัสดุ  
1     17:00  วิศวะ กลางวัน   
2     05:00   วิศวะ กลางคืน  
3     17:00          Office  
4     17:00      QC ตัดแต่ง  


C:\Users\chompoopan.jan\AppData\Local\Temp\ipykernel_33540\358935776.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  manpower["shift_start"] = pd.to_datetime(
C:\Users\chompoopan.jan\AppData\Local\Temp\ipykernel_33540\358935776.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  manpower["shift_end"] = pd.to_datetime(


In [7]:
# Cell 7: สร้าง attendance_today จาก master เป็นฐาน
# =========================

target_date = pd.to_datetime(TARGET_DATE).date()

# เอา attendance เฉพาะวันที่ต้องการ
attendance_scan_today = attendance[
    attendance["date"] == target_date
].copy()

# ใช้ master employee เป็นฐาน เพื่อให้คนไม่มี scan ก็ยังมีแถว
attendance_today = employee.merge(
    attendance_scan_today[["emp_id", "scan_name", "date", "scan_in", "scan_out"]],
    on="emp_id",
    how="left"
)

# ใส่วันที่ให้คนที่ไม่มี scan ด้วย
attendance_today["date"] = attendance_today["date"].fillna(target_date)

print(attendance_today.head())

     emp_id             name        name_key     status employment_type  \
0  10029242    ภัทรพงษ์ มงคล    ภัทรพงษ์มงคล  ทำงานอยู่        รายเดือน   
1  10039734   ศุภชัย สามัคคี   ศุภชัยสามัคคี  ทำงานอยู่        รายเดือน   
2  10150255  ชัยยันต์ ชุมกูล  ชัยยันต์ชุมกูล  ทำงานอยู่        รายเดือน   
3  10169542    ประไพ โพธิ์สี    ประไพโพธิ์สี  ทำงานอยู่        รายเดือน   
4  10191216    ศตวรรษ ต๊ะปวก    ศตวรรษต๊ะปวก  ทำงานอยู่        รายเดือน   

   position_code                                  position  \
0       10381588                       ผู้จัดการโรงงานสุกร   
1       10339613                       ผู้จัดการคลังสินค้า   
2       10297415    ผู้จัดการวางแผนและประสานงานการผลิต-ขาย   
3       10339091  ผู้ชำนาญการวางแผนและประสานงานการผลิต-ขาย   
4       10318635                    ผู้จัดการควบคุมการผลิต   

                           dept                               section  \
0    โรงงานชำแหละและตัดแต่งสุกร  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
1                 งานคลังสินค้

In [8]:
# =========================
# Cell 8: Detect Shift by Dept + Closest Shift Start
# =========================

import pandas as pd

# map กะจาก manpower ตาม dept
dept_shift_map = manpower_plan[
    ["dept", "shift", "shift_start"]
].dropna().drop_duplicates().copy()

dept_shift_map["dept_key"] = (
    dept_shift_map["dept"]
    .astype(str)
    .str.strip()
)

dept_shift_map["shift_start_dt"] = pd.to_datetime(
    dept_shift_map["shift_start"],
    format="%H:%M",
    errors="coerce"
)

dept_shift_map = dept_shift_map.dropna(subset=["shift_start_dt"])


def detect_shift_by_dept(row):

    if pd.isna(row["scan_in"]):
        return pd.Series({
            "shift": None,
            "shift_start": None,
            "shift_detect_reason": "Absent"
        })

    emp_dept = str(row["dept"]).strip()

    scan_time = pd.to_datetime(
        row["scan_in"],
        format="%H:%M",
        errors="coerce"
    )

    if pd.isna(scan_time):
        return pd.Series({
            "shift": None,
            "shift_start": None,
            "shift_detect_reason": "Invalid scan_in"
        })

    # เลือกเฉพาะกะของ dept เดียวกัน
    possible_shifts = dept_shift_map[
        dept_shift_map["dept_key"] == emp_dept
    ].copy()

    if possible_shifts.empty:
        return pd.Series({
            "shift": None,
            "shift_start": None,
            "shift_detect_reason": "No shift found for dept"
        })

    possible_shifts["diff_minutes"] = (
        possible_shifts["shift_start_dt"] - scan_time
    ).abs().dt.total_seconds() / 60

    closest = possible_shifts.sort_values("diff_minutes").iloc[0]

    return pd.Series({
        "shift": closest["shift"],
        "shift_start": closest["shift_start"],
        "shift_detect_reason": "Matched by dept closest time"
    })


attendance_today[["shift", "shift_start", "shift_detect_reason"]] = (
    attendance_today.apply(detect_shift_by_dept, axis=1)
)


def check_attendance_status(row):

    if pd.isna(row["scan_in"]):
        return "Absent"

    if pd.isna(row["shift_start"]):
        return "Unknown Shift"

    scan_in = pd.to_datetime(
        row["scan_in"],
        format="%H:%M",
        errors="coerce"
    )

    shift_start = pd.to_datetime(
        row["shift_start"],
        format="%H:%M",
        errors="coerce"
    )

    if pd.isna(scan_in) or pd.isna(shift_start):
        return "Invalid Time"

    late_time = shift_start + pd.Timedelta(minutes=LATE_GRACE_MINUTES)

    if scan_in > late_time:
        return "Late"
    else:
        return "Present"


attendance_today["attendance_status"] = attendance_today.apply(
    check_attendance_status,
    axis=1
)

available = attendance_today[
    attendance_today["attendance_status"].isin(["Present", "Late"])
].copy()

print(attendance_today[
    [
        "emp_id",
        "name",
        "dept",
        "scan_in",
        "shift",
        "shift_start",
        "attendance_status",
        "shift_detect_reason"
    ]
].head(30))

      emp_id                   name                          dept scan_in  \
0   10029242          ภัทรพงษ์ มงคล    โรงงานชำแหละและตัดแต่งสุกร     NaN   
1   10039734         ศุภชัย สามัคคี                 งานคลังสินค้า     NaN   
2   10150255        ชัยยันต์ ชุมกูล  งานวางแผนและประสานงานการผลิต     NaN   
3   10169542          ประไพ โพธิ์สี  งานวางแผนและประสานงานการผลิต     NaN   
4   10191216          ศตวรรษ ต๊ะปวก                  งานเชือดสุกร     NaN   
5   10232733      อภัตรชา เบ็ญจวรรณ                  งานเชือดสุกร     NaN   
6   10249292         นปภา แซ่เลี้ยว                  งานเชือดสุกร   04:38   
7   10251629          ภัทรพงษ์ ภากร                 งานคลังสินค้า     NaN   
8   10286270          กิ่งกมล มีศรี             งานเช็คจ่ายสินค้า     NaN   
9   10290692      ธิราทิพย์ นิยมทอง      งานแยกชิ้นส่วนและตัดแต่ง   00:35   
10  10291674          วราพร เขียวสี     งานสุขศาสตร์และภาชนะบรรจุ   04:50   
11  10296449          วรรณา นิลคูหา               งานตัดแต่งพิเศษ   02:03   

In [9]:
# 8.1 =========================
# สรุปจำนวนคนมา/สาย แยกตามหน่วยงาน
# =========================

dept_attendance_summary = (
    attendance_today
    .groupby(["dept", "attendance_status"])
    .agg(
        headcount=("emp_id", "nunique")
    )
    .reset_index()
)

print(dept_attendance_summary.head(30))

                            dept attendance_status  headcount
0                  งานคลังสินค้า            Absent         51
1                  งานคลังสินค้า              Late          5
2                  งานคลังสินค้า           Present         26
3                งานควบคุมคุณภาพ            Absent         14
4                งานควบคุมคุณภาพ           Present          6
5                งานตัดแต่งพิเศษ            Absent         65
6                งานตัดแต่งพิเศษ           Present         37
7                      งานผ่าซาก            Absent          2
8                      งานผ่าซาก           Present         18
9                       งานพัสดุ            Absent          2
10                      งานพัสดุ           Present          1
11                    งานรับสุกร            Absent          2
12                    งานรับสุกร           Present          4
13  งานวางแผนและประสานงานการผลิต            Absent          5
14  งานวางแผนและประสานงานการผลิต           Present          3
15      

In [10]:
# =========================
# Cell 9: Read Skill Matrix from Google Drive
# =========================

skill_matrix = pd.read_excel(SKILL_FILE)

skill_matrix = skill_matrix.rename(columns={
    "Employee ID": "emp_id",
    "Emp ID": "emp_id",
    "รหัสพนักงาน": "emp_id",
    "Skill": "skill",
    "ทักษะ": "skill",
    "Level": "level",
    "ระดับ": "level",
    "Can Do": "can_do",
    "ทำได้": "can_do"
})

skill_matrix["emp_id"] = skill_matrix["emp_id"].apply(clean_id)
skill_matrix["skill"] = skill_matrix["skill"].astype(str).str.strip()

if "level" not in skill_matrix.columns:
    skill_matrix["level"] = 3

if "can_do" not in skill_matrix.columns:
    skill_matrix["can_do"] = 1

skill_matrix["level"] = pd.to_numeric(skill_matrix["level"], errors="coerce").fillna(3).astype(int)
skill_matrix["can_do"] = pd.to_numeric(skill_matrix["can_do"], errors="coerce").fillna(1).astype(int)

skill_matrix = skill_matrix[
    ["emp_id", "skill", "level", "can_do"]
].copy()

print(skill_matrix.head())

     emp_id            skill  level  can_do
0  10029242           ออฟฟิศ      3       1
1  10039734           ออฟฟิศ      3       1
2  10150255           ออฟฟิศ      3       1
3  10169542  วางแผน Ordering      3       1
4  10249292        เชือดสุกร      3       1


In [ ]:
#print(
    scan_not_in_master[["emp_id", "Employee Name"]]
    .drop_duplicates()
    .head(30)
)

NameError: name 'scan_not_in_master' is not defined

In [15]:
#10
available_skill = available.merge(
    skill_matrix,
    on="emp_id",
    how="left",
    suffixes=("", "_skill")
)

print(available_skill.head())

     emp_id                name           name_key     status employment_type  \
0  10249292      นปภา แซ่เลี้ยว      นปภาแซ่เลี้ยว  ทำงานอยู่        รายเดือน   
1  10290692   ธิราทิพย์ นิยมทอง   ธิราทิพย์นิยมทอง  ทำงานอยู่        รายเดือน   
2  10291674       วราพร เขียวสี       วราพรเขียวสี  ทำงานอยู่        รายเดือน   
3  10296449       วรรณา นิลคูหา       วรรณานิลคูหา  ทำงานอยู่        รายเดือน   
4  10296630  อ้อยทิพย์ แจ้งสงค์  อ้อยทิพย์แจ้งสงค์  ทำงานอยู่        รายเดือน   

   position_code                                position  \
0       10318321  เจ้าหน้าที่ควบคุมการผลิต(เตรียมชำแหละ)   
1       10405230         ผู้ปฏิบัติการ (ตัดแต่งหมูลำตัว)   
2       10405217              ผู้ปฏิบัติการ (ล้างตะกร้า)   
3       10358618       ผู้ปฏิบัติการ (รับผลได้ ชิ้นส่วน)   
4       10405210         ผู้ปฏิบัติการ (คัดหมูซีกตลาดสด)   

                        dept                               section  \
0               งานเชือดสุกร  โรงงานชำแหละและตัดแต่งสุกรพระพุทธบาท   
1   งานแยกชิ

In [16]:
# =========================
# Cell 11: Create Options แบบเดิม ยังไม่ใช้ can_do
# =========================

options = []

for p_idx, plan_row in manpower_plan.iterrows():

    target_dept = str(plan_row["dept"]).strip()
    required_skill = str(plan_row["required_skill"]).strip()
    target_shift = str(plan_row["shift"]).strip()

    # คนที่มา + dept ตรง + shift ตรง
    dept_candidates = available[
        (available["dept"].astype(str).str.strip() == target_dept) &
        (available["shift"].astype(str).str.strip() == target_shift)
    ].copy()

    # merge skill
    dept_candidates = dept_candidates.merge(
        skill_matrix,
        on="emp_id",
        how="left"
    )

    dept_candidates["skill"] = dept_candidates["skill"].fillna("")

    dept_candidates["level"] = (
        dept_candidates["level"]
        .fillna(1)
        .astype(int)
    )

    # เช็ก skill ตรงกับ required_skill ไหม
    dept_candidates["skill_match"] = (
        dept_candidates["skill"]
        .astype(str)
        .str.strip()
        == required_skill
    )

    for _, emp in dept_candidates.iterrows():

        options.append({
            "plan_id": p_idx,
            "emp_id": emp["emp_id"],
            "name": emp["name"],
            "home_dept": emp["dept"],
            "target_dept": target_dept,
            "work_station": plan_row["work_station"],
            "shift": target_shift,
            "required_skill": required_skill,
            "emp_skill": emp["skill"],
            "skill_match": emp["skill_match"],
            "skill_level": emp["level"] if emp["skill_match"] else 1,
            "attendance_status": emp["attendance_status"],
            "scan_in": emp["scan_in"],
            "scan_out": emp["scan_out"],
            "home_dept_match": True
        })

options_df = pd.DataFrame(options)

print("options rows:", len(options_df))

if options_df.empty:
    print("❌ ไม่มี candidate เลย — เช็กว่า dept/shift ตรงกันไหม")

    print("available dept/shift:")
    display(
        available[["dept", "shift"]]
        .drop_duplicates()
        .head(100)
    )

    print("manpower dept/shift:")
    display(
        manpower_plan[["dept", "shift", "work_station", "required_hc"]]
        .drop_duplicates()
        .head(100)
    )

else:
    print("candidate emp:", options_df["emp_id"].nunique())
    display(options_df.head(30))

options rows: 1018
candidate emp: 224


,plan_id,emp_id,name,home_dept,target_dept,work_station,shift,required_skill,emp_skill,skill_match,skill_level,attendance_status,scan_in,scan_out,home_dept_match
0,0,10412311,เทพรังสรรค์ มั่นคง,งานพัสดุ,งานพัสดุ,พัสดุ,กะ 1,พัสดุ,พัสดุ,True,3,Present,06:59,06:59,True
1,3,10392242,นพพร คำจำรูญ,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,,False,1,Present,04:53,04:53,True
2,3,10539767,ธนนันท์ เดชเสริมศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,,False,1,Present,06:37,06:37,True
3,3,10539774,ศราวุธ สุทชา,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,QC คลังชิ้นส่วน,False,1,Present,01:56,01:56,True
4,3,10602863,กมลชนก ภูทอง,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,,False,1,Present,00:03,00:03,True
5,3,10621537,เบญจมาส โพธิ์ศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,QC ตัดแต่ง,False,1,Present,05:34,05:34,True
6,3,10688526,สุนิษา เฉลยมนต์,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,Office,กะ 1,Office,QC ตัดแต่ง,False,1,Present,04:30,04:30,True
7,4,10392242,นพพร คำจำรูญ,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,04:53,04:53,True
8,4,10539767,ธนนันท์ เดชเสริมศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,06:37,06:37,True
9,4,10539774,ศราวุธ สุทชา,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,QC คลังชิ้นส่วน,False,1,Present,01:56,01:56,True


In [17]:
manpower_skills = set(
    manpower_plan["required_skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

skill_skills = set(
    skill_matrix["skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

missing_required_skills = sorted(manpower_skills - skill_skills)

pd.DataFrame({
    "required_skill_missing_in_skill_matrix": missing_required_skills
})

,required_skill_missing_in_skill_matrix
0,QC คลังเครื่องใน
1,QC หมูเป็น
2,สุข นอกพื้นที่ แจกซักถุงมือ
3,สุข นอกพื้นที่ แต่งตัว ผลิต
4,สุข นอกพื้นที่ โรงอาหาร
5,สุุข ผลิต เชือด(1)


In [18]:
# เช็กว่าหน้างานไหนไม่มี candidate เลย
debug_rows = []

for p_idx, plan_row in manpower_plan.iterrows():
    required_skill = plan_row["required_skill"]
    required_hc = int(plan_row["required_hc"])

    candidates = options_df[options_df["plan_id"] == p_idx]

    debug_rows.append({
        "plan_id": p_idx,
        "dept": plan_row["dept"],
        "work_station": plan_row["work_station"],
        "required_skill": required_skill,
        "required_hc": required_hc,
        "candidate_count": len(candidates)
    })

debug_candidate = pd.DataFrame(debug_rows)
debug_candidate.sort_values("candidate_count").head(30)

,plan_id,dept,work_station,required_skill,required_hc,candidate_count
1,1,งานวิศวกรรม,วิศวะ กลางวัน,วิศวะ กลางวัน,3,0
2,2,งานวิศวกรรม,วิศวะ กลางคืน,วิศวะ กลางคืน,3,0
7,7,งานควบคุมคุณภาพ,QC คลังเครื่องใน,QC คลังเครื่องใน,1,0
5,5,งานควบคุมคุณภาพ,QC คลังชิ้นส่วน,QC คลังชิ้นส่วน,3,0
8,8,งานควบคุมคุณภาพ,QC คลังเครื่องใน,QC คลังเครื่องใน,1,0
27,27,งานคลังสินค้า,คลัง ตะกร้า,คลัง ตะกร้า,1,0
30,30,งานคลังสินค้า,คลัง จ่ายชิ้นส่วน,คลัง จ่ายชิ้นส่วน,9,0
28,28,งานคลังสินค้า,คลัง ตะกร้า,คลัง ตะกร้า,2,0
31,31,งานคลังสินค้า,คลัง จ่ายชิ้นส่วน,คลัง จ่ายชิ้นส่วน,8,0
25,25,งานคลังสินค้า,คลัง Picking,คลัง Picking,2,0


In [19]:
print("คนที่สแกนเข้าวันนี้:", attendance_today["emp_id"].nunique())
print("คนที่ available:", available["emp_id"].nunique())
print("จำนวนคนที่ manpower ต้องการทั้งหมด:", manpower_plan["required_hc"].sum())

คนที่สแกนเข้าวันนี้: 431
คนที่ available: 224
จำนวนคนที่ manpower ต้องการทั้งหมด: 357


In [20]:
# =========================
# Cell 12: Solver แบบบังคับเติมทุก plan_id เท่าที่มี candidate
# =========================

from pulp import LpProblem, LpVariable, LpMinimize, lpSum, LpBinary, LpStatus

model = LpProblem("Workforce_Allocation", LpMinimize)

x = {}

for i in options_df.index:
    x[i] = LpVariable(f"x_{i}", cat=LpBinary)

penalties = []

for i, row in options_df.iterrows():
    penalty = 0

    # มาสาย เลือกทีหลัง
    if row["attendance_status"] == "Late":
        penalty += 5

    # skill level สูง penalty ต่ำ
    #penalty += int(row["can_do_priority"]) * 100
    penalty += max(0, 5 - int(row["skill_level"]))

    # ถ้า skill ไม่ตรง เพิ่ม penalty แต่ยังจัดได้
    if "skill_match" in options_df.columns and row["skill_match"] == False:
        penalty += 20

    penalties.append(penalty * x[i])

model += lpSum(penalties)

# 1 คนจัดได้ไม่เกิน 1 จุดงาน
for emp_id in options_df["emp_id"].unique():
    emp_options = options_df[options_df["emp_id"] == emp_id].index
    model += lpSum(x[i] for i in emp_options) <= 1

# แต่ละ plan_id ต้องจัดเท่ากับ min(required_hc, candidate_count)
for p_idx, plan_row in manpower_plan.iterrows():
    required_hc = int(plan_row["required_hc"])

    plan_options = options_df[
        options_df["plan_id"] == p_idx
    ].index

    candidate_count = len(plan_options)
    target_hc = min(required_hc, candidate_count)

    model += lpSum(x[i] for i in plan_options) == target_hc

model.solve()

print("Solver Status:", LpStatus[model.status])

Solver Status: Infeasible


In [21]:
assigned_rows = []

for i, row in options_df.iterrows():
    if x[i].value() == 1:
        assigned_rows.append(row.to_dict())

allocation_result = pd.DataFrame(assigned_rows)

allocation_result.head()

,plan_id,emp_id,name,home_dept,target_dept,work_station,shift,required_skill,emp_skill,skill_match,skill_level,attendance_status,scan_in,scan_out,home_dept_match
0,4,10392242,นพพร คำจำรูญ,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,04:53,04:53,True
1,4,10539767,ธนนันท์ เดชเสริมศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,06:37,06:37,True
2,4,10539774,ศราวุธ สุทชา,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,QC คลังชิ้นส่วน,False,1,Present,01:56,01:56,True
3,4,10602863,กมลชนก ภูทอง,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,00:03,00:03,True
4,4,10621537,เบญจมาส โพธิ์ศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,QC ตัดแต่ง,True,3,Present,05:34,05:34,True


In [22]:
#13
assigned_rows = []

for i, row in options_df.iterrows():
    if x[i].value() == 1:
        assigned_rows.append(row.to_dict())

allocation_result = pd.DataFrame(assigned_rows)

allocation_result.head()

,plan_id,emp_id,name,home_dept,target_dept,work_station,shift,required_skill,emp_skill,skill_match,skill_level,attendance_status,scan_in,scan_out,home_dept_match
0,4,10392242,นพพร คำจำรูญ,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,04:53,04:53,True
1,4,10539767,ธนนันท์ เดชเสริมศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,06:37,06:37,True
2,4,10539774,ศราวุธ สุทชา,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,QC คลังชิ้นส่วน,False,1,Present,01:56,01:56,True
3,4,10602863,กมลชนก ภูทอง,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,,False,1,Present,00:03,00:03,True
4,4,10621537,เบญจมาส โพธิ์ศรี,งานควบคุมคุณภาพ,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,QC ตัดแต่ง,QC ตัดแต่ง,True,3,Present,05:34,05:34,True


In [23]:
#14
summary_rows = []

for p_idx, plan_row in manpower_plan.iterrows():
    required_hc = int(plan_row["required_hc"])

    if allocation_result.empty:
        assigned_hc = 0
    else:
        assigned_hc = len(allocation_result[allocation_result["plan_id"] == p_idx])

    summary_rows.append({
        "date": TARGET_DATE,
        "dept": plan_row["dept"],
        "work_station": plan_row["work_station"],
        "shift": plan_row["shift"],
        "required_hc": required_hc,
        "assigned_hc": assigned_hc,
        "gap": assigned_hc - required_hc
    })

gap_summary = pd.DataFrame(summary_rows)

gap_summary.head()

,date,dept,work_station,shift,required_hc,assigned_hc,gap
0,2026-05-19,งานพัสดุ,พัสดุ,กะ 1,3,0,-3
1,2026-05-19,งานวิศวกรรม,วิศวะ กลางวัน,กะ 1,3,0,-3
2,2026-05-19,งานวิศวกรรม,วิศวะ กลางคืน,กะ 2,3,0,-3
3,2026-05-19,งานควบคุมคุณภาพ,Office,กะ 1,1,0,-1
4,2026-05-19,งานควบคุมคุณภาพ,QC ตัดแต่ง,กะ 1,7,6,-1


In [24]:
# =========================
# Rebalance within same dept + same shift
# โยกคนจาก work_station ที่ gap > 0
# ไปเติม work_station ที่ gap < 0
# เฉพาะ dept เดียวกัน และกะเดียวกัน
# =========================

allocation_result = allocation_result.copy()

if "allocation_type" not in allocation_result.columns:
    allocation_result["allocation_type"] = "Solver Allocation"

move_rows = []

# จุดงานที่ขาดคน
shortage_ws = gap_summary[gap_summary["gap"] < 0].copy()
shortage_ws["shortage_hc"] = shortage_ws["gap"].abs()

# จุดงานที่คนเกิน
surplus_ws = gap_summary[gap_summary["gap"] > 0].copy()
surplus_ws["surplus_hc"] = surplus_ws["gap"]

for _, shortage in shortage_ws.iterrows():

    target_dept = str(shortage["dept"]).strip()
    target_ws = str(shortage["work_station"]).strip()
    target_shift = str(shortage["shift"]).strip()
    need_hc = int(shortage["shortage_hc"])

    # หา work_station ที่คนเกินใน dept เดียวกัน และกะเดียวกัน
    donor_list = surplus_ws[
        (surplus_ws["dept"].astype(str).str.strip() == target_dept) &
        (surplus_ws["shift"].astype(str).str.strip() == target_shift)
    ].copy()

    for donor_idx, donor in donor_list.iterrows():

        if need_hc <= 0:
            break

        donor_ws = str(donor["work_station"]).strip()
        donor_shift = str(donor["shift"]).strip()
        surplus_hc = int(surplus_ws.loc[donor_idx, "surplus_hc"])

        if surplus_hc <= 0:
            continue

        donor_people = allocation_result[
            (allocation_result["target_dept"].astype(str).str.strip() == target_dept) &
            (allocation_result["work_station"].astype(str).str.strip() == donor_ws) &
            (allocation_result["shift"].astype(str).str.strip() == target_shift)
        ].copy()

        if donor_people.empty:
            continue

        move_count = min(need_hc, surplus_hc, len(donor_people))
        selected_people = donor_people.head(move_count)

        for idx, person in selected_people.iterrows():

            new_row = person.copy()

            new_row["work_station"] = target_ws
            new_row["target_dept"] = target_dept
            new_row["shift"] = target_shift
            new_row["allocation_type"] = f"Moved from {donor_ws} to {target_ws} within same shift"

            move_rows.append(new_row)

            allocation_result = allocation_result.drop(index=idx)

        need_hc -= move_count
        surplus_ws.loc[donor_idx, "surplus_hc"] -= move_count

if move_rows:
    move_df = pd.DataFrame(move_rows)
    allocation_result = pd.concat(
        [allocation_result, move_df],
        ignore_index=True
    )

print("จำนวนคนที่โยกเติมจาก work_station ที่เกินใน dept และกะเดียวกัน:", len(move_rows))

จำนวนคนที่โยกเติมจาก work_station ที่เกินใน dept และกะเดียวกัน: 0


In [25]:
# =========================
# Recalculate Gap Summary หลังโยกคน
# =========================

summary_rows = []

for p_idx, plan_row in manpower_plan.iterrows():

    required_hc = int(plan_row["required_hc"])

    assigned_hc = len(
        allocation_result[
            (allocation_result["target_dept"].astype(str).str.strip() == str(plan_row["dept"]).strip()) &
            (allocation_result["work_station"].astype(str).str.strip() == str(plan_row["work_station"]).strip()) &
            (allocation_result["shift"].astype(str).str.strip() == str(plan_row["shift"]).strip())
        ]
    )

    summary_rows.append({
        "date": TARGET_DATE,
        "dept": plan_row["dept"],
        "work_station": plan_row["work_station"],
        "shift": plan_row["shift"],
        "required_hc": required_hc,
        "assigned_hc": assigned_hc,
        "gap": assigned_hc - required_hc,
        "shortage": max(required_hc - assigned_hc, 0),
        "surplus": max(assigned_hc - required_hc, 0)
    })

gap_summary = pd.DataFrame(summary_rows)

gap_summary.sort_values("shortage", ascending=False).head()

,date,dept,work_station,shift,required_hc,assigned_hc,gap,shortage,surplus
43,2026-05-19,งานเครื่องใน,เครื่องในขาว,กะ 2,12,0,-12,12,0
49,2026-05-19,งานเชือดสุกร,เชือดสุกร,กะ 2,12,0,-12,12,0
30,2026-05-19,งานคลังสินค้า,คลัง จ่ายชิ้นส่วน,กะ 2,9,0,-9,9,0
57,2026-05-19,งานตัดแต่งพิเศษ,ไหล่(พิเศษ),กะ 2,9,0,-9,9,0
47,2026-05-19,งานเช็คจ่ายสินค้า,เช็คจ่ายสินค้า,กะ 2,8,0,-8,8,0


In [26]:
# =========================
# Cell 15: Export Result
# =========================

# -------------------------
# สร้าง Dept Attendance Summary
# -------------------------

dept_attendance_summary = (
    attendance_today
    .groupby(["dept", "attendance_status"])
    .agg(
        headcount=("emp_id", "nunique")
    )
    .reset_index()
)

dept_attendance_pivot = (
    dept_attendance_summary
    .pivot_table(
        index="dept",
        columns="attendance_status",
        values="headcount",
        fill_value=0
    )
    .reset_index()
)

# กันกรณีบาง status ไม่มี
for col in ["Present", "Late", "Absent"]:
    if col not in dept_attendance_pivot.columns:
        dept_attendance_pivot[col] = 0

dept_attendance_pivot["Total_Come"] = (
    dept_attendance_pivot["Present"] +
    dept_attendance_pivot["Late"]
)

dept_attendance_pivot = dept_attendance_pivot[
    [
        "dept",
        "Present",
        "Late",
        "Total_Come",
        "Absent"
    ]
]

dept_attendance_pivot = dept_attendance_pivot.sort_values(
    "Total_Come",
    ascending=False
)

# -------------------------
# หา dept ที่ไม่มี scan
# -------------------------

all_depts = (
    employee["dept"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

all_depts = pd.DataFrame({
    "dept": all_depts
})

scan_depts = (
    scan_with_dept[
        scan_with_dept["Timestamp"].notna()
    ]["dept"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

scan_depts = pd.DataFrame({
    "dept": scan_depts
})

no_scan_depts = all_depts[
    ~all_depts["dept"].isin(scan_depts["dept"])
].copy()

no_scan_depts = no_scan_depts.sort_values("dept")

# -------------------------
# ตั้งชื่อ output file
# -------------------------

date_text = TARGET_DATE.replace("-", "")

output_file = os.path.join(
    OUTPUT_FOLDER,
    f"workforce_allocation_result_{date_text}.xlsx"
)

# -------------------------
# Export Excel
# -------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    allocation_result.to_excel(
        writer,
        sheet_name="Allocation Result",
        index=False
    )

    gap_summary.to_excel(
        writer,
        sheet_name="Gap Summary",
        index=False
    )

    attendance_today.to_excel(
        writer,
        sheet_name="Attendance Status",
        index=False
    )

    manpower_plan.to_excel(
        writer,
        sheet_name="Manpower Plan Clean",
        index=False
    )

    dept_attendance_pivot.to_excel(
        writer,
        sheet_name="Dept Attendance Summary",
        index=False
    )

    no_scan_depts.to_excel(
        writer,
        sheet_name="Dept No Scan",
        index=False
    )

    scan_with_dept.to_excel(
        writer,
        sheet_name="Timestamp With Dept",
        index=False
    )

print("Exported:", output_file)

Exported: C:\Users\chompoopan.jan\OneDrive - Charoen Pokphand Foods Group\PARINYA THEPHI's files - PLP - Process Improvement (Production)\3_Step 3\Production System\workforce_system\Output\workforce_allocation_result_20260519.xlsx
